# UNSUPERVISED PART : CVRIE

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split

print('Numpy:', np.__version__)
print('Pandas:', pd.__version__)

Numpy: 2.4.2
Pandas: 3.0.1


In [2]:
csv_path = "../Student_Dataset.csv"
dataframe = pd.read_csv(csv_path, encoding='utf-8-sig')
print(dataframe.head())
print(dataframe.shape)
print(dataframe.info())


                                                   1  \
2                                           0x000000   
3                                           0x000000   
4  A short sprint brought on a twinge in the midd...   
5                                           0x000000   
6  A sudden twist while reaching for a bag made o...   

  On uneven paths the heel feels wobbly side‑to‑side. Slanted pavements bring a quick aching shift under the ankle when I step.  
2  The feverish feeling eased, but I’m now breath...                                                                             
3  While working in the garden, I touched a metal...                                                                             
4                                                NaN                                                                             
5  I accidentally left my hand on a hot mug for a...                                                                             
6            

# EDA 

## Analyse préliminaire :

on peut constater que ce dataset contient 1010 lignes et 2 colonnes mais lorsque'on regarde le cvs de plus pret on peut voir des hexadecimal parfois :

C'est la conséquence des virgules dans les témoignages qui sont mal parser donc il faut corriger ca.

In [6]:
csv_path = "../Student_Dataset.csv"

tempdf = pd.read_csv(csv_path, encoding='utf-8-sig', header=None,
    names=['id', 'col2', 'col3'])

tempdf['text'] = np.where(tempdf['col2'] == '0x000000', tempdf['col3'], tempdf['col2'])
dataframe = tempdf[['id', 'text']]



print(dataframe.head())
print(dataframe.shape)
print(dataframe.info())
print('nombres elements louche dans le df: ', dataframe['text'].str.contains('0x000000').sum())


   id                                               text
0   1  On uneven paths the heel feels wobbly side‑to‑...
1   2  The feverish feeling eased, but I’m now breath...
2   3  While working in the garden, I touched a metal...
3   4  A short sprint brought on a twinge in the midd...
4   5  I accidentally left my hand on a hot mug for a...
(1011, 2)
<class 'pandas.DataFrame'>
RangeIndex: 1011 entries, 0 to 1010
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   id      1011 non-null   int64
 1   text    1011 non-null   str  
dtypes: int64(1), str(1)
memory usage: 15.9 KB
None
nombres elements louche dans le df:  0


Maitenant que nous avons enlever la 3 eme colonne aléatoire a cause du mauvais parsing du csv on peut commencer a s'occuper des stop words : Les motes qui n'ont pas d'utilité pour lié les différents témoiganges en eux :


# Les Stops Words

Comme les témopignages sont en anglais on peut utilisé la liste de stopwords contenue dans nltk (environ 180 mots) et on convertie en unordered collection pour la rapidité des lookups

In [8]:
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def remove_stop_words(text):
    tokens = text.lower().split()
    tokens = [t for t in tokens if t not in stop_words]
    return ' '.join(tokens)

dataframe['clean_text'] = dataframe['text'].apply(remove_stop_words)

print(dataframe[['text', 'clean_text']].head(3))
print('\nExemple avant :', dataframe['text'][0])
print('Exemple après  :', dataframe['clean_text'][0])


                                                text  \
0  On uneven paths the heel feels wobbly side‑to‑...   
1  The feverish feeling eased, but I’m now breath...   
2  While working in the garden, I touched a metal...   

                                          clean_text  
0  uneven paths heel feels wobbly side‑to‑side. s...  
1  feverish feeling eased, i’m breathing faster d...  
2  working garden, touched metal grill stove. lef...  

Exemple avant : On uneven paths the heel feels wobbly side‑to‑side. Slanted pavements bring a quick aching shift under the ankle when I step.
Exemple après  : uneven paths heel feels wobbly side‑to‑side. slanted pavements bring quick aching shift ankle step.


[nltk_data] Downloading package stopwords to /home/vanity/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


ici on crée une nouvelle colonne clean_text car on veut grader la colonne original pour éventuels debug .